In [ ]:
# Cell 1 — Upload zip file directly to Colab
from google.colab import files
import zipfile
import os

print("Click 'Choose Files' and select zip_file.zip")
uploaded = files.upload()

# Unzip to raw_stocks folder
print("\nUnzipping...")
with zipfile.ZipFile('zip_file.zip', 'r') as z:
    z.extractall('/content/raw_stocks')
    extracted = z.namelist()
    print(f"Extracted {len(extracted)} files:")
    for f in extracted:
        print(f"  {f}")

Click 'Choose Files' and select zip_file.zip



Unzipping...
Extracted 21 files:
  zip_file/
  zip_file/AXISBANK_minute.csv
  zip_file/BAJAJ-AUTO_minute.csv
  zip_file/CIPLA_minute.csv
  zip_file/DIVISLAB_minute.csv
  zip_file/DRREDDY_minute.csv
  zip_file/EICHERMOT_minute.csv
  zip_file/HCLTECH_minute.csv
  zip_file/HDFCBANK_minute.csv
  zip_file/HEROMOTOCO_minute.csv
  zip_file/ICICIBANK_minute.csv
  zip_file/INFY_minute.csv
  zip_file/KOTAKBANK_minute.csv
  zip_file/MARUTI_minute.csv
  zip_file/SBIN_minute.csv
  zip_file/SUNPHARMA_minute.csv
  zip_file/TCS_minute.csv
  zip_file/TECHM_minute.csv
  zip_file/TORNTPHARM_minute.csv
  zip_file/TVSMOTOR_minute.csv
  zip_file/WIPRO_minute.csv


In [ ]:
# ============================================================
# FIX 1 — ROBUST DATE PARSER
# Handles mixed formats in same file
# Some files have YYYY-DD-MM, some have DD-MM-YYYY
# Some switch formats mid-file at row 3375
# ============================================================

def parse_date_column(series):
    """
    Robustly parse date column that may have mixed formats.

    Detected formats in your data:
      Format A: DD-MM-YYYY HH:MM     e.g. 02-02-2015 09:15
      Format B: YYYY-DD-MM HH:MM:SS  e.g. 2015-13-02 09:15:00
      Mixed:    Some files switch format at row 3375

    Strategy: try each format, fall back to mixed parsing
    """
    # Try Format A first: DD-MM-YYYY HH:MM (your original format)
    try:
        parsed = pd.to_datetime(
            series,
            format='%d-%m-%Y %H:%M',
            dayfirst=True
        )
        return parsed
    except Exception:
        pass

    # Try Format A with seconds: DD-MM-YYYY HH:MM:SS
    try:
        parsed = pd.to_datetime(
            series,
            format='%d-%m-%Y %H:%M:%S',
            dayfirst=True
        )
        return parsed
    except Exception:
        pass

    # Try Format B: YYYY-MM-DD HH:MM:SS (standard ISO)
    try:
        parsed = pd.to_datetime(
            series,
            format='%Y-%m-%d %H:%M:%S'
        )
        return parsed
    except Exception:
        pass

    # Try Format B without seconds: YYYY-MM-DD HH:MM
    try:
        parsed = pd.to_datetime(
            series,
            format='%Y-%m-%d %H:%M'
        )
        return parsed
    except Exception:
        pass

    # Last resort: mixed format parsing row by row
    # Slowest but handles any inconsistency
    print("    Using mixed format parsing (file has "
          "inconsistent date formats)...")
    parsed = pd.to_datetime(
        series,
        format='mixed',
        dayfirst=True
    )
    return parsed


# ============================================================
# FIX 2 — UPDATE SECTOR MAP WITH CORRECT FILENAMES
# Edit this after seeing Step 0 output
# Note: TVSMOTOR appeared — update accordingly
# ============================================================

# UPDATE THIS based on Step 0 output
# Current best guess based on your output
SECTOR_MAP = {
    'Banking': [
        'HDFCBANK_minute.csv',
        'ICICIBANK_minute.csv',
        'KOTAKBANK_minute.csv',
        'AXISBANK_minute.csv',
        'SBIN_minute.csv',
    ],
    'IT': [
        'TCS_minute.csv',
        'INFY_minute.csv',
        'WIPRO_minute.csv',
        'HCLTECH_minute.csv',
        'TECHM_minute.csv',
    ],
    'Pharma': [
        'SUNPHARMA_minute.csv',
        'DRREDDY_minute.csv',
        'CIPLA_minute.csv',
        'DIVISLAB_minute.csv',
        'TORNTPHARM_minute.csv',
    ],
    'Auto': [
        'TVSMOTOR_minute.csv',   # ← appeared in your output
        'MARUTI_minute.csv', # ← check if this exists
        'BAJAJ-AUTO_minute.csv',
        'HEROMOTOCO_minute.csv',
        'EICHERMOT_minute.csv',
    ]
}

ALL_FILES = [f for files in SECTOR_MAP.values() for f in files]

Path(TRIM_DIR).mkdir(parents=True, exist_ok=True)
Path(FEAT_DIR).mkdir(parents=True, exist_ok=True)

print(f"\nConfigured for {len(ALL_FILES)} stocks")


# ============================================================
# FIXED LOAD AND TRIM
# ============================================================

def load_and_trim(filepath, start_dt, end_dt):
    """
    Load one stock CSV and trim to date range.
    Uses robust date parser that handles mixed formats.
    """
    df = pd.read_csv(filepath)

    # Lowercase columns
    df.columns = [c.lower().strip() for c in df.columns]

    # Find date column
    date_col = None
    for col in df.columns:
        if any(x in col for x in
               ['date', 'time', 'datetime', 'timestamp']):
            date_col = col
            break

    if date_col is None:
        raise ValueError(
            f"No date column found. Columns: {list(df.columns)}"
        )

    if date_col != 'date':
        df = df.rename(columns={date_col: 'date'})

    # Parse with robust parser
    df['date'] = parse_date_column(df['date'])

    df = df.set_index('date').sort_index()

    # Keep OHLCV only
    ohlcv = ['open', 'high', 'low', 'close', 'volume']
    df    = df[[c for c in ohlcv if c in df.columns]]

    # Trim
    return df.loc[start_dt:end_dt]


# ============================================================
# FIXED FEATURE ENGINEERING
# Fix: bar_index must be numpy array not pandas Index
# ============================================================

def engineer_features(df):
    """
    Engineer 10 features from trimmed OHLCV.

    FIX: bar_index computed as numpy array to avoid
    'Index has no attribute clip' error.
    """
    feat = pd.DataFrame(index=df.index)

    # 1. Intrabar log return
    feat['log_return'] = np.log(
        df['close'] / df['open'].replace(0, np.nan)
    )

    # 2. High-Low range
    feat['hl_range'] = np.log(
        df['high'] / df['low'].replace(0, np.nan)
    )

    # 3. Volume normalized
    vol_mean = df['volume'].rolling(
        window=20, min_periods=1
    ).mean()
    feat['vol_normalized'] = df['volume'] / (vol_mean + 1e-8)

    # 4-6. Lagged returns
    feat['ret_lag1']  = feat['log_return'].shift(1)
    feat['ret_lag5']  = feat['log_return'].shift(5)
    feat['ret_lag15'] = feat['log_return'].shift(15)

    # 7-8. Rolling volatility
    feat['rolling_vol5']  = feat['log_return'].rolling(
        window=5, min_periods=1
    ).std()
    feat['rolling_vol20'] = feat['log_return'].rolling(
        window=20, min_periods=1
    ).std()

    # 9-10. Intraday time encoding
    # FIX: extract hour/minute as numpy arrays first
    hours   = df.index.hour.to_numpy()    # numpy array
    minutes = df.index.minute.to_numpy()  # numpy array

    # Minutes since market open (9:15 AM)
    bar_index = (hours * 60 + minutes - 9 * 60 - 15)
    bar_index = np.clip(bar_index, 0, 374)  # numpy clip — works

    feat['time_sin'] = np.sin(2 * np.pi * bar_index / 375)
    feat['time_cos'] = np.cos(2 * np.pi * bar_index / 375)

    # Target: close price 5 bars ahead
    feat['target_5min'] = df['close'].shift(-5)

    # Keep close for reference
    feat['close'] = df['close']

    return feat


# ============================================================
# TRIM ALL STOCKS
# ============================================================

def trim_all_stocks():

    start_dt = pd.Timestamp(START_DATETIME)
    end_dt   = pd.Timestamp(END_DATETIME)

    print("=" * 75)
    print("STEP 1: TRIMMING")
    print(f"  From: {start_dt}  →  To: {end_dt}")
    print("=" * 75)

    print(
        f"\n{'Stock':<16} {'Raw Rows':>10} {'Trimmed':>10} "
        f"{'First Bar':<22} {'Last Bar':<22} Status"
    )
    print("-" * 90)

    results = []

    for filename in ALL_FILES:
        ticker  = filename.replace('_minute.csv', '')
        inpath  = os.path.join(INPUT_DIR, filename)
        outpath = os.path.join(TRIM_DIR, filename)

        if not os.path.exists(inpath):
            print(f"{ticker:<16} ✗ FILE NOT FOUND: {filename}")
            results.append({
                'ticker': ticker, 'status': 'FILE NOT FOUND',
                'trimmed_rows': 0
            })
            continue

        try:
            raw_rows = sum(1 for _ in open(inpath)) - 1
            df_t     = load_and_trim(inpath, start_dt, end_dt)
            n        = len(df_t)

            if n == 0:
                status = "⚠ NO DATA"
            else:
                first = df_t.index[0]
                last  = df_t.index[-1]
                gap_s = abs((first - start_dt).days)
                gap_e = abs((last  - end_dt).days)
                status = (
                    "✓ OK" if gap_s <= 5 and gap_e <= 5
                    else f"⚠ CHECK DATES"
                )
                df_t.reset_index().to_csv(outpath, index=False)
                print(
                    f"{ticker:<16} {raw_rows:>10,} {n:>10,} "
                    f"{str(first):<22} {str(last):<22} {status}"
                )

            results.append({
                'ticker': ticker, 'status': status,
                'trimmed_rows': n
            })

        except Exception as e:
            print(f"{ticker:<16} ✗ ERROR: {str(e)[:80]}")
            results.append({
                'ticker': ticker,
                'status': f'ERROR: {str(e)[:60]}',
                'trimmed_rows': 0
            })

    df_r = pd.DataFrame(results)
    ok   = df_r[df_r['status'] == '✓ OK']
    print(f"\nTrimmed OK: {len(ok)} / {len(ALL_FILES)}")

    if len(ok) > 0:
        rows = ok['trimmed_rows'].astype(int)
        print(f"Row counts — "
              f"Min: {rows.min():,}  "
              f"Max: {rows.max():,}  "
              f"Mean: {rows.mean():,.0f}")

    problems = df_r[df_r['status'] != '✓ OK']
    if len(problems) > 0:
        print(f"\n⚠ Problem stocks:")
        for _, r in problems.iterrows():
            print(f"  {r['ticker']}: {r['status']}")

    df_r.to_csv(
        os.path.join(TRIM_DIR, '_trim_report.csv'),
        index=False
    )
    return df_r


# ============================================================
# FEATURIZE ALL STOCKS
# ============================================================

def featurize_all_stocks():

    FEATURE_COLS = [
        'log_return', 'hl_range', 'vol_normalized',
        'ret_lag1', 'ret_lag5', 'ret_lag15',
        'rolling_vol5', 'rolling_vol20',
        'time_sin', 'time_cos'
    ]

    print("\n" + "=" * 75)
    print("STEP 2: FEATURE ENGINEERING")
    print("=" * 75)

    print(
        f"\n{'Stock':<16} {'Input Rows':>12} "
        f"{'Output Rows':>12} {'Dropped':>10} Status"
    )
    print("-" * 60)

    results = []

    for filename in ALL_FILES:
        ticker  = filename.replace('_minute.csv', '')
        inpath  = os.path.join(TRIM_DIR, filename)
        outpath = os.path.join(FEAT_DIR, filename)

        if not os.path.exists(inpath):
            print(f"{ticker:<16} ✗ TRIMMED FILE NOT FOUND")
            results.append({
                'ticker': ticker, 'status': 'NOT TRIMMED',
                'output_rows': 0
            })
            continue

        try:
            df = pd.read_csv(inpath)
            df.columns = [c.lower().strip() for c in df.columns]
            df['date']  = parse_date_column(df['date'])
            df = df.set_index('date').sort_index()

            input_rows = len(df)
            feat_df    = engineer_features(df)

            # Drop NaN from lag/rolling and missing target
            feat_df = feat_df.dropna(
                subset=FEATURE_COLS + ['target_5min']
            )

            output_rows = len(feat_df)
            dropped     = input_rows - output_rows

            feat_df.reset_index().to_csv(outpath, index=False)

            print(
                f"{ticker:<16} {input_rows:>12,} "
                f"{output_rows:>12,} {dropped:>10,} ✓ OK"
            )

            results.append({
                'ticker':      ticker,
                'input_rows':  input_rows,
                'output_rows': output_rows,
                'dropped':     dropped,
                'status':      '✓ OK'
            })

        except Exception as e:
            print(f"{ticker:<16} ✗ ERROR: {str(e)[:80]}")
            results.append({
                'ticker': ticker,
                'status': f'ERROR: {str(e)[:60]}',
                'output_rows': 0
            })

    df_r = pd.DataFrame(results)
    ok   = df_r[df_r['status'] == '✓ OK']
    print(f"\nFeaturized OK: {len(ok)} / {len(ALL_FILES)}")

    if len(ok) > 0:
        rows = ok['output_rows'].astype(int)
        print(f"Row counts — "
              f"Min: {rows.min():,}  "
              f"Max: {rows.max():,}  "
              f"Mean: {rows.mean():,.0f}")

    df_r.to_csv(
        os.path.join(FEAT_DIR, '_feature_report.csv'),
        index=False
    )
    return df_r


# ============================================================
# SANITY CHECK
# ============================================================

def sanity_check():
    print("\n" + "=" * 75)
    print("STEP 3: SANITY CHECK")
    print("=" * 75)

    feat_csvs = sorted([
        f for f in os.listdir(FEAT_DIR)
        if f.endswith('.csv') and not f.startswith('_')
    ])

    print(f"\nFeature files created: {len(feat_csvs)}")

    if len(feat_csvs) == 0:
        print("No feature files found — check errors above")
        return

    # Load one and inspect
    sample  = feat_csvs[0]
    ticker  = sample.replace('_minute.csv', '')
    df_chk  = pd.read_csv(
        os.path.join(FEAT_DIR, sample),
        nrows=100
    )

    print(f"\nSample stock: {ticker}")
    print(f"Shape: {df_chk.shape}")
    print(f"Columns: {list(df_chk.columns)}")
    print(f"\nFirst 3 rows:")
    print(df_chk.head(3).to_string())

    nan_counts = df_chk.isnull().sum()
    if nan_counts.sum() == 0:
        print(f"\n✓ No NaN in first 100 rows")
    else:
        print(f"\n⚠ NaN found:")
        print(nan_counts[nan_counts > 0])

    print(f"\n✓ Pipeline complete")
    print(f"  Trimmed files : {TRIM_DIR}")
    print(f"  Feature files : {FEAT_DIR}")
    print(f"\nNext step: train/val/test split and model training")


# ============================================================
# RUN EVERYTHING
# ============================================================

trim_results = trim_all_stocks()
feat_results = featurize_all_stocks()
sanity_check()


Configured for 20 stocks
STEP 1: TRIMMING
  From: 2015-02-02 09:15:00  →  To: 2026-01-22 15:29:00

Stock              Raw Rows    Trimmed First Bar              Last Bar               Status
------------------------------------------------------------------------------------------
HDFCBANK          1,033,704  1,014,954 2015-02-02 09:15:00    2026-01-22 15:29:00    ✓ OK
ICICIBANK         1,033,704  1,014,954 2015-02-02 09:15:00    2026-01-22 15:29:00    ✓ OK
KOTAKBANK         1,033,702  1,014,952 2015-02-02 09:15:00    2026-01-22 15:29:00    ✓ OK
AXISBANK          1,033,600  1,014,960 2015-02-02 09:15:00    2026-01-22 15:29:00    ✓ OK
SBIN              1,033,703  1,014,953 2015-02-02 09:15:00    2026-01-22 15:29:00    ✓ OK
TCS               1,033,700  1,014,950 2015-02-02 09:15:00    2026-01-22 15:29:00    ✓ OK
INFY              1,033,707  1,014,957 2015-02-02 09:15:00    2026-01-22 15:29:00    ✓ OK
WIPRO             1,033,697  1,014,947 2015-02-02 09:15:00    2026-01-22 15:29:00    ✓ 

In [ ]:
# ============================================================
# CELL 1 — ALL IMPORTS AND CONFIGURATION
# Run this first after every restart
# ============================================================

import pandas as pd
import numpy as np
import os
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

# Paths
FEAT_DIR = "/content/feature_stocks"
TRIM_DIR = "/content/trimmed_stocks"

# Split boundaries
TRAIN_END = '2021-12-31 15:30'
VAL_END   = '2022-12-31 15:30'

# Model config
SEQ_LEN    = 30
N_FEATURES = 10
N_STOCKS   = 20
BATCH_SIZE = 512

# Stock universe
SECTOR_MAP = {
    'Banking': ['HDFCBANK', 'ICICIBANK', 'KOTAKBANK',
                'AXISBANK', 'SBIN'],
    'IT':      ['TCS', 'INFY', 'WIPRO',
                'HCLTECH', 'TECHM'],
    'Pharma':  ['SUNPHARMA', 'DRREDDY', 'CIPLA',
                'DIVISLAB', 'TORNTPHARM'],
    'Auto':    ['MARUTI', 'TVSMOTOR', 'BAJAJ-AUTO',
                'HEROMOTOCO', 'EICHERMOT']
}

ALL_TICKERS = [
    t for tickers in SECTOR_MAP.values()
    for t in tickers
]

FEATURE_COLS = [
    'log_return', 'hl_range', 'vol_normalized',
    'ret_lag1', 'ret_lag5', 'ret_lag15',
    'rolling_vol5', 'rolling_vol20',
    'time_sin', 'time_cos'
]

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print(f"Device:   {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU:      {torch.cuda.get_device_name(0)}")
print(f"Stocks:   {len(ALL_TICKERS)}")
print(f"Features: {len(FEATURE_COLS)} per stock")

# Verify files exist
missing = [
    t for t in ALL_TICKERS
    if not os.path.exists(
        os.path.join(FEAT_DIR, f"{t}_minute.csv")
    )
]
if missing:
    print(f"\n⚠ Missing: {missing}")
    print("Re-run feature engineering pipeline first")
else:
    print(f"\n✓ All 20 feature files confirmed")

Device:   cuda
GPU:      NVIDIA A100-SXM4-80GB
Stocks:   20
Features: 10 per stock

✓ All 20 feature files confirmed


In [ ]:
# ============================================================
# CELL 2 — LOAD ALL STOCKS (fixed: deduplication on load)
# ============================================================

def parse_date_column(series):
    for fmt in [
        '%d-%m-%Y %H:%M', '%d-%m-%Y %H:%M:%S',
        '%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M'
    ]:
        try:
            return pd.to_datetime(series, format=fmt)
        except Exception:
            continue
    return pd.to_datetime(
        series, format='mixed', dayfirst=True
    )

def load_all_stocks():
    stock_data = {}
    print("Loading feature files...")
    for ticker in ALL_TICKERS:
        path = os.path.join(
            FEAT_DIR, f"{ticker}_minute.csv"
        )
        df = pd.read_csv(path)
        df.columns = [c.lower().strip() for c in df.columns]
        df['date'] = parse_date_column(df['date'])
        df = df.set_index('date').sort_index()

        # Remove duplicates
        n_before = len(df)
        df = df[~df.index.duplicated(keep='last')]
        n_after  = len(df)

        removed = n_before - n_after
        suffix  = f" (removed {removed} dupes)" \
                  if removed > 0 else ""
        print(f"  ✓ {ticker:<15} {n_after:>10,} rows{suffix}")
        stock_data[ticker] = df

    print(f"\nLoaded: {len(stock_data)} / {N_STOCKS}")
    return stock_data

stock_data = load_all_stocks()

Loading feature files...
  ✓ HDFCBANK         1,014,932 rows (removed 2 dupes)
  ✓ ICICIBANK        1,010,417 rows (removed 2 dupes)
  ✓ KOTAKBANK        1,014,930 rows (removed 2 dupes)
  ✓ AXISBANK         1,014,938 rows (removed 2 dupes)
  ✓ SBIN             1,013,791 rows (removed 2 dupes)
  ✓ TCS              1,014,540 rows
  ✓ INFY             1,010,422 rows
  ✓ WIPRO            1,014,927 rows
  ✓ HCLTECH          1,014,932 rows
  ✓ TECHM            1,014,926 rows
  ✓ SUNPHARMA        1,014,924 rows
  ✓ DRREDDY          1,014,937 rows
  ✓ CIPLA            1,014,939 rows
  ✓ DIVISLAB         1,014,936 rows
  ✓ TORNTPHARM       1,014,899 rows
  ✓ MARUTI           1,014,931 rows
  ✓ TVSMOTOR         1,014,923 rows (removed 2 dupes)
  ✓ BAJAJ-AUTO       1,014,938 rows (removed 2 dupes)
  ✓ HEROMOTOCO       1,014,932 rows (removed 2 dupes)
  ✓ EICHERMOT        1,014,931 rows (removed 2 dupes)

Loaded: 20 / 20


In [ ]:
# ============================================================
# CELL 3 — ALIGN ALL STOCKS (fixed: reindex not loc)
# ============================================================

def align_stocks(stock_data):
    print("Aligning timestamps...")
    indices    = {t: set(df.index)
                  for t, df in stock_data.items()}
    common_idx = set.intersection(*indices.values())
    common_idx = pd.DatetimeIndex(sorted(common_idx))

    print(f"  Common timestamps: {len(common_idx):,}")
    print(f"  Range: {common_idx[0]} → {common_idx[-1]}")

    aligned = {}
    for ticker, df in stock_data.items():
        df_a  = df.reindex(common_idx)
        n_nan = df_a.isnull().sum().sum()
        if n_nan > 0:
            df_a = df_a.ffill().bfill()
            print(f"  ⚠ {ticker}: filled {n_nan} NaN")
        assert len(df_a) == len(common_idx), \
            f"Alignment failed for {ticker}"
        aligned[ticker] = df_a

    print(f"  ✓ All {len(aligned)} stocks aligned")
    return aligned, common_idx

aligned_data, common_idx = align_stocks(stock_data)

Aligning timestamps...
  Common timestamps: 1,008,787
  Range: 2015-02-02 09:30:00 → 2026-01-22 15:24:00
  ✓ All 20 stocks aligned


In [ ]:
# ============================================================
# CELL 4 — TEMPORAL SPLIT
# ============================================================

def temporal_split(common_idx):
    train_mask = common_idx <= pd.Timestamp(TRAIN_END)
    val_mask   = (
        (common_idx > pd.Timestamp(TRAIN_END)) &
        (common_idx <= pd.Timestamp(VAL_END))
    )
    test_mask  = common_idx > pd.Timestamp(VAL_END)

    train_idx = common_idx[train_mask]
    val_idx   = common_idx[val_mask]
    test_idx  = common_idx[test_mask]
    total     = len(common_idx)

    print("TEMPORAL SPLIT")
    print("-" * 55)
    print(f"  Train: {len(train_idx):>8,}  "
          f"{train_idx[0].date()} → {train_idx[-1].date()}  "
          f"({len(train_idx)/total*100:.1f}%)")
    print(f"  Val:   {len(val_idx):>8,}  "
          f"{val_idx[0].date()} → {val_idx[-1].date()}  "
          f"({len(val_idx)/total*100:.1f}%)")
    print(f"  Test:  {len(test_idx):>8,}  "
          f"{test_idx[0].date()} → {test_idx[-1].date()}  "
          f"({len(test_idx)/total*100:.1f}%)")
    return train_idx, val_idx, test_idx

train_idx, val_idx, test_idx = temporal_split(common_idx)

TEMPORAL SPLIT
-------------------------------------------------------
  Train:  633,000  2015-02-02 → 2021-12-31  (62.7%)
  Val:     92,669  2022-01-03 → 2022-12-30  (9.2%)
  Test:   283,118  2023-01-02 → 2026-01-22  (28.1%)


In [ ]:
# ============================================================
# CELL 5 — BUILD FEATURE MATRICES AND NORMALIZE
# ============================================================

def build_feature_matrix(aligned_data, timestamps):
    feature_blocks = []
    target_blocks  = []
    for ticker in ALL_TICKERS:
        df   = aligned_data[ticker].loc[timestamps]
        feat = df[FEATURE_COLS].values
        feature_blocks.append(feat)
        target_blocks.append(df['target_5min'].values)
    X = np.concatenate(feature_blocks, axis=1)
    Y = np.stack(target_blocks, axis=1)
    nan_X = np.isnan(X).sum()
    nan_Y = np.isnan(Y).sum()
    if nan_X > 0 or nan_Y > 0:
        print(f"  ⚠ NaN X:{nan_X} Y:{nan_Y} — filling")
        X = np.nan_to_num(X, nan=0.0)
        Y = np.nan_to_num(Y, nan=0.0)
    return X, Y

print("Building feature matrices...")
X_train, Y_train = build_feature_matrix(
    aligned_data, train_idx)
X_val,   Y_val   = build_feature_matrix(
    aligned_data, val_idx)
X_test,  Y_test  = build_feature_matrix(
    aligned_data, test_idx)

print(f"  X_train: {X_train.shape}  Y_train: {Y_train.shape}")
print(f"  X_val:   {X_val.shape}")
print(f"  X_test:  {X_test.shape}")

# Normalize — fit on TRAIN only
scaler      = StandardScaler()
X_train     = scaler.fit_transform(X_train)
X_val       = scaler.transform(X_val)
X_test      = scaler.transform(X_test)

target_mean = Y_train.mean(axis=0)
target_std  = Y_train.std(axis=0)
target_std  = np.where(target_std == 0, 1.0, target_std)

Y_train_norm = (Y_train - target_mean) / target_std
Y_val_norm   = (Y_val   - target_mean) / target_std
Y_test_norm  = (Y_test  - target_mean) / target_std

print(f"\n✓ Normalization done (train stats only)")
print(f"✓ Price range: {target_mean.min():.2f} → "
      f"{target_mean.max():.2f}")

Building feature matrices...
  X_train: (633000, 200)  Y_train: (633000, 20)
  X_val:   (92669, 200)
  X_test:  (283118, 200)

✓ Normalization done (train stats only)
✓ Price range: 131.89 → 6456.04


In [ ]:
# ============================================================
# CELL 6 — DATASET AND DATALOADERS
# ============================================================

class StockSequenceDataset(Dataset):
    def __init__(self, X, Y, seq_len=SEQ_LEN):
        self.X       = torch.FloatTensor(X)
        self.Y       = torch.FloatTensor(Y)
        self.seq_len = seq_len
        self.n       = len(X) - seq_len

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        x = self.X[idx : idx + self.seq_len]
        y = self.Y[idx + self.seq_len]
        return x, y

train_dataset = StockSequenceDataset(X_train, Y_train_norm)
val_dataset   = StockSequenceDataset(X_val,   Y_val_norm)
test_dataset  = StockSequenceDataset(X_test,  Y_test_norm)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=2, pin_memory=True
)

print(f"Train sequences: {len(train_dataset):,}")
print(f"Val sequences:   {len(val_dataset):,}")
print(f"Test sequences:  {len(test_dataset):,}")
print(f"Batch size:      {BATCH_SIZE}")

Train sequences: 632,970
Val sequences:   92,639
Test sequences:  283,088
Batch size:      512


In [ ]:
# ============================================================
# CELL 7 — MODEL COMPONENTS: FIR + LGHI
# ============================================================

class Learnable_FIR(nn.Module):
    def __init__(self, in_channels, filter_len=15):
        super().__init__()
        self.filter_len  = filter_len
        self.in_channels = in_channels
        self.low_pass  = nn.Parameter(
            torch.randn(in_channels, 1, filter_len) * 0.01
        )
        self.high_pass = nn.Parameter(
            torch.randn(in_channels, 1, filter_len) * 0.01
        )

    def forward(self, x):
        x_t = x.transpose(1, 2)
        pad  = self.filter_len // 2
        low  = nn.functional.conv1d(
            x_t, self.low_pass,
            padding=pad, groups=self.in_channels
        )[:, :, :x_t.shape[2]]
        high = nn.functional.conv1d(
            x_t, self.high_pass,
            padding=pad, groups=self.in_channels
        )[:, :, :x_t.shape[2]]
        return low.transpose(1, 2), high.transpose(1, 2)


class LGHI(nn.Module):
    def __init__(self, d_model, n_heads=4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k     = d_model // n_heads
        self.W_Q  = nn.Linear(d_model, d_model)
        self.W_K  = nn.Linear(d_model, d_model)
        self.W_V  = nn.Linear(d_model, d_model)
        self.W_O  = nn.Linear(d_model, d_model)
        self.gamma = nn.Parameter(torch.tensor(-5.0))
        self.norm  = nn.LayerNorm(d_model)

    def forward(self, L, H):
        B, T, D = L.shape
        Q = self.W_Q(L).view(
            B, T, self.n_heads, self.d_k
        ).transpose(1, 2)
        K = self.W_K(L).view(
            B, T, self.n_heads, self.d_k
        ).transpose(1, 2)
        V = self.W_V(H).view(
            B, T, self.n_heads, self.d_k
        ).transpose(1, 2)
        att = torch.softmax(
            torch.matmul(Q, K.transpose(-2,-1))
            / (self.d_k ** 0.5),
            dim=-1
        )
        Z = torch.matmul(att, V)
        Z = Z.transpose(1,2).contiguous().view(B, T, D)
        Z = self.W_O(Z)
        Y = L + torch.sigmoid(self.gamma) * Z
        return self.norm(Y)

print("✓ Learnable_FIR and LGHI defined")

✓ Learnable_FIR and LGHI defined


In [ ]:
# ============================================================
# CELL 8 — DTWAtt (vectorized — no Python loops)
# ============================================================

class DTWAtt(nn.Module):
    """
    Vectorized DTWAtt.
    Learns a universal feature sequence U and measures
    alignment of each time step to it via L2 distance
    combined with a learned projection.
    Fully GPU-parallelized — no Python for loops.
    """
    def __init__(self, d_model, seq_len,
                 window=4, n_heads=5):
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.window  = window
        self.n_heads = n_heads

        # Learnable universal template per head
        self.U = nn.Parameter(
            torch.randn(n_heads, d_model) * 0.01
        )

        # Learned alignment projection
        self.alignment = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.Tanh(),
            nn.Linear(d_model // 2, n_heads)
        )

    def forward(self, V):
        # V: (batch, T, d_model)
        U_exp = self.U.unsqueeze(0).unsqueeze(0)
        # (1, 1, n_heads, d_model)

        V_exp = V.unsqueeze(2)
        # (batch, T, 1, d_model)

        # L2 distance to each universal template
        dist = torch.norm(V_exp - U_exp, dim=-1)
        # (batch, T, n_heads)

        # Learned alignment scores
        align = self.alignment(V)
        # (batch, T, n_heads)

        return align - dist
        # (batch, T, n_heads)

print("✓ DTWAtt defined")

✓ DTWAtt defined


In [ ]:
# ============================================================
# CELL 9 — FULL PROPOSED MODEL
# ============================================================

class LGHI_DTWAtt_LSTM(nn.Module):
    def __init__(
        self,
        input_dim,
        d_model=128,
        seq_len=SEQ_LEN,
        n_stocks=N_STOCKS,
        lstm_layers=2,
        dtw_window=4,
        dtw_heads=5,
        dropout=0.3
    ):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.fir      = Learnable_FIR(d_model, 15)
        self.lghi     = LGHI(d_model, n_heads=4)
        self.dtw_att  = DTWAtt(
            d_model, seq_len, dtw_window, dtw_heads
        )
        self.dtw_proj = nn.Linear(dtw_heads, d_model)
        self.lstm = nn.LSTM(
            input_size  = d_model * 2,
            hidden_size = d_model,
            num_layers  = lstm_layers,
            batch_first = True,
            dropout     = dropout if lstm_layers > 1 else 0
        )
        self.output_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, n_stocks)
        )

    def forward(self, x):
        x_proj    = self.input_proj(x)
        low, high = self.fir(x_proj)
        fused     = self.lghi(low, high)
        dtw_out   = self.dtw_att(fused)
        dtw_feat  = self.dtw_proj(dtw_out)
        combined  = torch.cat([fused, dtw_feat], dim=-1)
        lstm_out, _ = self.lstm(combined)
        pred      = self.output_head(lstm_out[:, -1, :])
        return pred

print("✓ LGHI_DTWAtt_LSTM defined")

✓ LGHI_DTWAtt_LSTM defined


In [ ]:
# ============================================================
# CELL 10 — METRICS
# ============================================================

def compute_metrics(y_true, y_pred, prefix=""):
    r2_list, mse_list, rmse_list, mae_list, mape_list \
        = [], [], [], [], []

    for i in range(y_true.shape[1]):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        r2_list.append(r2_score(yt, yp))
        mse = mean_squared_error(yt, yp)
        mse_list.append(mse)
        rmse_list.append(np.sqrt(mse))
        mae_list.append(mean_absolute_error(yt, yp))
        mask = np.abs(yt) > 1e-8
        mape_list.append(
            np.mean(np.abs(
                (yt[mask]-yp[mask])/yt[mask]
            ))*100 if mask.sum()>0 else np.nan
        )

    return {
        f'{prefix}R2':   np.mean(r2_list),
        f'{prefix}MSE':  np.mean(mse_list),
        f'{prefix}RMSE': np.mean(rmse_list),
        f'{prefix}MAE':  np.mean(mae_list),
        f'{prefix}MAPE': np.nanmean(mape_list),
    }, {
        'r2':   r2_list,
        'rmse': rmse_list,
        'mae':  mae_list
    }

def print_metrics(metrics, title=""):
    print(f"\n{title}")
    print("-" * 40)
    for k, v in metrics.items():
        print(f"  {k:<12}: {v:.6f}")

print("✓ Metrics defined")

✓ Metrics defined


In [ ]:
# ============================================================
# CELL 11 — TRAINING FUNCTION
# ============================================================

def train_proposed_model(
    model, train_loader, val_loader,
    n_epochs=50, lr=3e-5, patience=10
):
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=1e-5
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5
    )
    criterion = nn.MSELoss()

    best_val     = float('inf')
    patience_ctr = 0
    train_losses = []
    val_losses   = []

    print(f"\nTraining on {DEVICE}")
    print(f"  Epochs: {n_epochs}  LR: {lr}  "
          f"Patience: {patience}")
    print("-" * 55)
    print(f"{'Epoch':>6} {'Train Loss':>12} "
          f"{'Val Loss':>12} {'Status':>12}")
    print("-" * 55)

    for epoch in range(1, n_epochs + 1):

        # Train
        model.train()
        total_loss = 0.0
        n_batches  = 0
        for X_b, Y_b in train_loader:
            X_b = X_b.to(DEVICE)
            Y_b = Y_b.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_b), Y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )
            optimizer.step()
            total_loss += loss.item()
            n_batches  += 1
        avg_train = total_loss / n_batches

        # Validate
        model.eval()
        total_val = 0.0
        n_val     = 0
        with torch.no_grad():
            for X_b, Y_b in val_loader:
                X_b = X_b.to(DEVICE)
                Y_b = Y_b.to(DEVICE)
                total_val += criterion(
                    model(X_b), Y_b
                ).item()
                n_val += 1
        avg_val = total_val / n_val

        train_losses.append(avg_train)
        val_losses.append(avg_val)
        scheduler.step(avg_val)

        if avg_val < best_val:
            best_val     = avg_val
            patience_ctr = 0
            torch.save(
                model.state_dict(),
                '/content/best_model.pt'
            )
            status = "✓ saved"
        else:
            patience_ctr += 1
            status = f"patience {patience_ctr}/{patience}"

        print(f"{epoch:>6} {avg_train:>12.6f} "
              f"{avg_val:>12.6f} {status:>12}")

        if patience_ctr >= patience:
            print(f"\nEarly stopping at epoch {epoch}")
            break

    model.load_state_dict(
        torch.load('/content/best_model.pt')
    )
    print(f"\nBest val loss: {best_val:.6f}")
    return model, train_losses, val_losses

print("✓ Training function defined")

✓ Training function defined


In [ ]:
# ============================================================
# CELL 12 — INFERENCE FUNCTION
# ============================================================

def get_deep_predictions(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X_b, Y_b in loader:
            preds.append(
                model(X_b.to(DEVICE)).cpu().numpy()
            )
            targets.append(Y_b.numpy())
    preds   = np.concatenate(preds,   axis=0)
    targets = np.concatenate(targets, axis=0)
    return (
        preds   * target_std + target_mean,
        targets * target_std + target_mean
    )

print("✓ Inference function defined")


✓ Inference function defined


In [ ]:
# ============================================================
# CELL 13 — BASELINE DATA PREP AND MODELS
# ============================================================

def prepare_baseline_data(X, Y, seq_len=SEQ_LEN):
    n = len(X) - seq_len
    X_flat = np.zeros((n, seq_len * X.shape[1]))
    Y_flat = np.zeros((n, Y.shape[1]))
    for i in range(n):
        X_flat[i] = X[i:i+seq_len].flatten()
        Y_flat[i] = Y[i+seq_len]
    return X_flat, Y_flat

print("Preparing baseline data...")
X_train_flat, Y_train_flat = prepare_baseline_data(
    X_train, Y_train_norm)
X_val_flat,   Y_val_flat   = prepare_baseline_data(
    X_val, Y_val_norm)
X_test_flat,  Y_test_flat  = prepare_baseline_data(
    X_test, Y_test_norm)
print(f"X_train_flat: {X_train_flat.shape}")
print(f"X_test_flat:  {X_test_flat.shape}")


def train_linear_regression():
    print("\n" + "=" * 55)
    print("BASELINE 1: Linear Regression")
    print("=" * 55)
    m = LinearRegression(n_jobs=-1)
    m.fit(X_train_flat, Y_train_flat)
    pred_d   = m.predict(X_test_flat) \
               * target_std + target_mean
    target_d = Y_test_flat * target_std + target_mean
    met, _   = compute_metrics(
        target_d, pred_d, prefix="LR_"
    )
    print_metrics(met, "Linear Regression — Test")
    return m, met, pred_d, target_d


def train_knn():
    print("\n" + "=" * 55)
    print("BASELINE 2: KNN (k=5)")
    print("=" * 55)
    X_knn = X_train_flat[-50000:]
    Y_knn = Y_train_flat[-50000:]
    print(f"  Using 50,000 most recent samples")
    m = KNeighborsRegressor(
        n_neighbors=5, metric='euclidean', n_jobs=-1
    )
    m.fit(X_knn, Y_knn)
    pred_d   = m.predict(X_test_flat) \
               * target_std + target_mean
    target_d = Y_test_flat * target_std + target_mean
    met, _   = compute_metrics(
        target_d, pred_d, prefix="KNN_"
    )
    print_metrics(met, "KNN — Test")
    return m, met, pred_d, target_d


def train_random_forest():
    print("\n" + "=" * 55)
    print("BASELINE 3: Random Forest")
    print("=" * 55)
    m = RandomForestRegressor(
        n_estimators=200, max_depth=8,
        min_samples_leaf=100,
        max_features='sqrt',
        n_jobs=-1, random_state=42
    )
    print(f"  Training on {len(X_train_flat):,} samples...")
    m.fit(X_train_flat, Y_train_flat)
    pred_d   = m.predict(X_test_flat) \
               * target_std + target_mean
    target_d = Y_test_flat * target_std + target_mean
    met, _   = compute_metrics(
        target_d, pred_d, prefix="RF_"
    )
    print_metrics(met, "Random Forest — Test")
    return m, met, pred_d, target_d

print("✓ Baseline functions defined")


Preparing baseline data...
X_train_flat: (632970, 6000)
X_test_flat:  (283088, 6000)
✓ Baseline functions defined


In [ ]:
# ============================================================
# CELL 14 — TRAIN ALL MODELS
# ============================================================

# Initialize proposed model
print("=" * 65)
print("INITIALIZING PROPOSED MODEL")
print("=" * 65)

proposed_model = LGHI_DTWAtt_LSTM(
    input_dim   = N_STOCKS * N_FEATURES,
    d_model     = 128,
    seq_len     = SEQ_LEN,
    n_stocks    = N_STOCKS,
    lstm_layers = 2,
    dtw_window  = 4,
    dtw_heads   = 5,
    dropout     = 0.3
).to(DEVICE)

total_params = sum(
    p.numel() for p in proposed_model.parameters()
    if p.requires_grad
)
print(f"Parameters: {total_params:,}")

# Train proposed model
proposed_model, train_losses, val_losses = \
    train_proposed_model(
        proposed_model, train_loader, val_loader,
        n_epochs=50, lr=3e-5, patience=10
    )

# Proposed model test predictions
print("\nRunning inference on test set...")
prop_preds, prop_targets = get_deep_predictions(
    proposed_model, test_loader
)
prop_metrics, _ = compute_metrics(
    prop_targets, prop_preds, prefix="PROP_"
)
print_metrics(prop_metrics, "LGHI+DTWAtt+LSTM — Test")

# Baselines
lr_model,  lr_metrics,  lr_preds,  lr_targets  = \
    train_linear_regression()
knn_model, knn_metrics, knn_preds, knn_targets = \
    train_knn()
rf_model,  rf_metrics,  rf_preds,  rf_targets  = \
    train_random_forest()

INITIALIZING PROPOSED MODEL
Parameters: 445,146

Training on cuda
  Epochs: 50  LR: 3e-05  Patience: 10
-------------------------------------------------------
 Epoch   Train Loss     Val Loss       Status
-------------------------------------------------------
     1     0.540215     1.367210      ✓ saved
     2     0.492383     0.961224      ✓ saved
     3     0.459745     0.837594      ✓ saved
     4     0.300403     0.797897      ✓ saved
     5     0.268915     0.802826 patience 1/10
     6     0.247798     0.797337      ✓ saved
     7     0.239428     0.815968 patience 1/10
     8     0.221587     0.807055 patience 2/10
     9     0.213118     0.820775 patience 3/10
    10     0.206813     0.820274 patience 4/10
    11     0.236312     0.824284 patience 5/10
    12     0.255116     0.834762 patience 6/10
    13     0.259753     0.835719 patience 7/10
    14     0.294967     0.795908      ✓ saved
    15     0.323018     0.776267      ✓ saved
    16     0.298086     0.788799 patienc

KeyboardInterrupt: 

In [ ]:
# Interrupt the current cell first (click stop button)
# Then run this

from sklearn.linear_model import Ridge
from sklearn.linear_model import SGDRegressor
from sklearn.multioutput import MultiOutputRegressor
import numpy as np

def train_linear_regression_fast():
    print("\n" + "=" * 55)
    print("BASELINE 1: Linear Regression (Ridge/SGD)")
    print("=" * 55)

    # Use SGD-based linear regression
    # Much faster than OLS on large datasets
    # Mathematically equivalent for our purpose
    print(f"  Training on {len(X_train_flat):,} samples...")
    print(f"  Input dim: {X_train_flat.shape[1]}")

    # Ridge regression — handles high-dim better than OLS
    # alpha=1.0 is light regularization
    model = Ridge(alpha=1.0, fit_intercept=True)

    # Fit one model per stock target
    # MultiOutputRegressor wraps Ridge for multi-output
    multi_model = MultiOutputRegressor(
        model, n_jobs=-1  # parallel across stocks
    )

    print("  Fitting (parallel across 20 stocks)...")
    multi_model.fit(X_train_flat, Y_train_flat)

    pred_norm     = multi_model.predict(X_test_flat)
    pred_denorm   = pred_norm   * target_std + target_mean
    target_denorm = Y_test_flat * target_std + target_mean

    metrics, _ = compute_metrics(
        target_denorm, pred_denorm, prefix="LR_"
    )
    print_metrics(metrics, "Linear Regression — Test")
    return multi_model, metrics, pred_denorm, target_denorm

# Run this first
lr_model, lr_metrics, lr_preds, lr_targets = \
    train_linear_regression_fast()


BASELINE 1: Linear Regression (Ridge/SGD)
  Training on 632,970 samples...
  Input dim: 6000
  Fitting (parallel across 20 stocks)...
